Exploración de AgentRisk

In [2]:
import pandas as pd

In [5]:
train_path = "../data/raw/data/train.jsonl"
dev_path = "../data/raw/data/dev.jsonl"
split_path = "../data/raw/metadata/split_manifest.jsonl"

In [9]:
train = pd.read_json(train_path, lines=True)
dev = pd.read_json(dev_path, lines=True)
split = pd.read_json(split_path, lines=True)
print(len(train))
print(len(dev))
print(len(split))

568
143
711


In [12]:
print("--- TRAIN ---")
print(train.columns.tolist())

print("\n--- DEV ---")
print(dev.columns.tolist())

print("\n--- MANIFEST ---")
print(split.columns.tolist())

--- TRAIN ---
['example_id', 'formatted_interaction', 'proposed_action', 'risk_severity', 'risk_domains', 'risk_drivers']

--- DEV ---
['example_id', 'formatted_interaction', 'proposed_action', 'risk_severity', 'risk_domains', 'risk_drivers']

--- MANIFEST ---
['example_id', 'split', 'scenario_group']


Los conteos de filas coinciden exactamente con la ficha del dataset: 568 (train) + 143 (dev) = 711 (manifest).

In [ ]:
#Comparar example_ids del manifest con train y dev juntos
train_dev_ids = set(train['example_id']).union(set(dev['example_id']))
manifest_ids = set(split['example_id'])
print("¿Coinciden exactamente los IDs del manifest con train y dev?", train_dev_ids == manifest_ids)

#Comprobar que el split del manifest coincide con cada conjunto
train_match = set(train['example_id']).issubset(set(split[split['split'] == 'train']['example_id']))
dev_match = set(dev['example_id']).issubset(set(split[split['split'] == 'dev']['example_id']))
print("¿El split 'train' coincide?", train_match)
print("¿El split 'dev' coincide?", dev_match)

#Contar scenario_groups distintos y sus frecuencias
print("\nCantidad de scenario_group distintos:", split['scenario_group'].nunique())
print("\nConteo por cada scenario_group:")
print(split['scenario_group'].value_counts())

¿Coinciden exactamente los IDs del manifest con train y dev? True
¿El split 'train' coincide? True
¿El split 'dev' coincide? True

Cantidad de scenario_group distintos: 397

Conteo por cada scenario_group:
scenario_group
G000    41
G001    36
G002    17
G003    13
G004    11
        ..
G392     1
G393     1
G394     1
G395     1
G396     1
Name: count, Length: 397, dtype: int64


In [15]:
#Conteo por nivel de risk_severity ordenado de 1 a 5
print("--- TRAIN: risk_severity ---")
print(train['risk_severity'].value_counts().sort_index())

print("\n--- DEV: risk_severity ---")
print(dev['risk_severity'].value_counts().sort_index())

#Media y mediana de train (la mediana es tu L0 para S1)
mean_s1 = train['risk_severity'].mean()
median_s1 = train['risk_severity'].median()

print(f"\nMedia en train (S1): {mean_s1:.2f}")
print(f"Mediana en train (S1) [Tu L0]: {median_s1}")

--- TRAIN: risk_severity ---
risk_severity
1    265
2     33
3    127
4    136
5      7
Name: count, dtype: int64

--- DEV: risk_severity ---
risk_severity
1    66
2     8
3    32
4    35
5     2
Name: count, dtype: int64

Media en train (S1): 2.27
Mediana en train (S1) [Tu L0]: 2.0


In [16]:
#Desplegar (explode) y contar frecuencias en train
print("--- TOP DOMINIOS (risk_domains) ---")
domains_counts = train['risk_domains'].explode().value_counts()
print(domains_counts)
print(f"-> Más frecuente (L0 para S2a): {domains_counts.index[0]}")

print("\n--- TOP FACTORES (risk_drivers) ---")
drivers_counts = train['risk_drivers'].explode().value_counts()
print(drivers_counts)
print(f"-> Más frecuente (L0 para S2b): {drivers_counts.index[0]}")

--- TOP DOMINIOS (risk_domains) ---
risk_domains
No Material Harm                         262
Privacy / Confidentiality                180
Security / System Integrity               65
Financial / Assets                        53
Physical / Health                         52
Operational / Service                     44
Legal / Compliance                        15
Reputational / Social / Psychological      6
Information Quality                        3
Environmental                              2
Discrimination / Fairness                  1
Name: count, dtype: int64
-> Más frecuente (L0 para S2a): No Material Harm

--- TOP FACTORES (risk_drivers) ---
risk_drivers
No Material Risk                        262
Untrusted / Conflicting Instructions    195
Authorization / Consent Failure         128
Scope / Safeguard Failure               108
Insufficient Grounding / Uncertainty     62
Execution / Action Hazard                26
Risky Requested Outcome                  14
Name: count, dtype: int